# Calcul des performances
- Évaluation  

## Importations
- codecs pour les encodages
- pandas et numpy pour les calculs sur tableaux
- matplotlib pour les graphiques
- itertools pour les itérateurs sophistiqués (paires sur liste, ...)

In [1]:
# -*- coding: utf8 -*-
import re,codecs
import pandas as pd
import numpy as np
import yaml

def ding():
    os.system('afplay /System/Library/Sounds/Submarine.aiff')

In [2]:
import yaml

In [3]:
from IPython.display import display, HTML

In [4]:
import datetime
def dateheure():
    return datetime.datetime.utcnow().strftime('%y%m%d%H%M')

In [5]:
saut="\n"

# Choix de l'échantillon et du gold
- *inputType* est le type de l'échantillon de départ
    - CVk-Type pour les k-fold (k le nombre de morceaux)
        - Type=Train pour le k-fold standard
        - Type=Test pour le k-fold inverse
    - S pour les déciles, 8 pour les 10%, 7 pour les 20%, 6 pour les 30%, 5 pour les 40%, 4 pour les 50%
- *checkType* est le type de contrôle pour les formes
    - gold pour un contrôle correspondant aux formes attestées
    - platinum pour un contrôle correspondant à l'ensemble théorique des formes
- *num* est le numéro de l'input considéré

### Dédoubler les lignes avec des surabondances dans *colonne*
>identifier une ligne avec surabondance

>>ajouter les lignes correspondant à chaque valeur

>>ajouter le numéro de la ligne initiale dans les lignes à supprimer

>supprimer les lignes avec surabondance

NB : il faut préparer le tableau pour avoir une indexation qui permette l'ajout des valeurs individuelles et la suppression des lignes de surabondances

# Lecture de l'échantillon

In [6]:
# neutralisationsNORD=(u"6û",u"9ê")
# neutralisationsSUD=(u"e2o",u"E9O")
# if phonologicalMap=="-N":
#     neutralisations=neutralisationsNORD
# elif phonologicalMap=="-S":
#     neutralisations=neutralisationsSUD
# else:
#     neutralisations=(u"",u"")
#     phonologicalMap=("-X")
# bdlexiqueIn = u"èò"+neutralisations[0]
# bdlexiqueNum = [ord(char) for char in bdlexiqueIn]
# neutreOut = u"EO"+neutralisations[1]
# neutralise = dict(zip(bdlexiqueNum, neutreOut))

neutralisationsTotales=(u"e2o6û",u"E9O9ê")
totalNeutreIn=u"èò"+neutralisationsTotales[0]
totalNeutreNum=[ord(char) for char in totalNeutreIn]
totalNeutreOut=u"EO"+neutralisationsTotales[1]
totalNeutralise = dict(zip(totalNeutreNum, totalNeutreOut))

In [7]:
def recoder(chaine,table=totalNeutralise):
    if type(chaine)==str:
        temp=chaine.translate(table)
        result=temp
    elif type(chaine)==unicode:
        result=chaine.translate(table)
    else:
        result=chaine
    return result

### Vérification de la phonotactique des glides du français
- si *prononciation* est *None* renvoyer *None*
- ajout de diérèses dans les séquences mal-formées
- vérification des séquences consonne+glide à la finale

In [8]:
dierese={"j":"ij", "w":"uw","H":"yH","i":"ij","u":"uw","y":"yH"}
glide2voc={"j":"i","w":"u","H":"y"}

In [9]:
def checkFrench(prononciation):
    if prononciation and not pd.isnull(prononciation):
        result=recoder(prononciation)
        # Consonne plus glide final
        m=re.match(r"^(.*[^ieèEaOouy926êôâ])([jwH])$",result)
        if m:
            # print ("pb avec un glide final", [prononciation])
            result=m.group(1)+glide2voc[m.group(2)]
        # attaque Obs+Liq+Glide
        m=re.match(r"(.*[ptkbdgfsSvzZ][rl])([jwH])(.*)",result)
        if m:
            n=re.search(r"[ptkbdgfsSvzZ][rl](wa|Hi|wê)",result)
            if not n:
                glide=m.group(2)
                result=m.group(1)+dierese[glide]+m.group(3)
        # Voyelle haute+Voyelle => diérèse
        m=re.match(r"(.*)([iuy])([ieEaOouy].*)",result)
        if m:
            glide=m.group(2)
            result=m.group(1)+dierese[glide]+m.group(3)
        # yod ou n palatal+yod
        m=re.match(r"(.*)([jJ])(j)(.*)",result)
        if m:
            result=m.group(1)+m.group(2)+m.group(4)
            # print(prononciation,"=>",result)
        m=re.match(r"^(.*[^ieèEaOouy926êôâ])([jwH])6(.*)$",result)
        if m:
            result=m.group(1)+glide2voc[m.group(2)]+m.group(3)
            # print(prononciation,"=>",result)
    else:
        result=prononciation
    return result

In [10]:
def flattenData(tData):
    dfData=tData.melt("lexeme",var_name="cell",value_name="form")
    dfData=dfData.dropna(thresh=3).set_index(["lexeme", "cell"]).apply(lambda x: x.str.split(',').explode()).sort_values("lexeme").reset_index()
    return dfData
    
def separateData(df1,df2):
    df1=pd.merge(df1,df2,on=["lexeme","cell","form"],how="left",indicator=True)
    df1=df1.loc[df1["_merge"]=="left_only"].drop("_merge", axis=1)
    return df1

In [11]:
def getTableDf(file):
    table=pd.read_csv(file,sep=";",encoding="utf8")
    if u"Unnamed: 0" in table.columns:
        del table[u"Unnamed: 0"]
    table=table.dropna(axis=1,how='all')
    # print(len(table.columns))
    df=flattenData(table)
    df.form=df.form.apply(lambda x: checkFrench(x))
    return table,df

le tableau tOutput contient la sortie de SWIM avec toutes les formes (initiales et générées) sous forme de paradigmes  
- les surabondances sont dans la même case séparées par une virgule dans tOutput

le tableau dfOutput contient seulement les formes générées, une par ligne  
- les surabondances sont sur deux lignes différentes dans dfOutput

- sampleCases pour la liste des cases effectivement représentées dans le corpus de départ 

# Identifier les lexèmes potentiellement générés

In [12]:
def identifyLexemes():
    testLexemes=[]
    outLexemes={}
    for ix,row in tInput.iloc[:,:].iterrows():
        if row.lexeme in testLexemes:display(row.dropna())
        if ix%100==0: print(ix,end=", ")
        dCriteres=row.dropna().to_dict()
        del dCriteres["lexeme"]
        if row.lexeme in testLexemes:print(dCriteres)
        selCriteres=[]
        for k,v in dCriteres.items():
            if "," in v:
                lVs=v.split(",")
                for lV in lVs:
                    selCriteres.append("((dfComplete.cell=='%s') & (dfComplete.form=='%s'))"%(k,lV))
            else:
                selCriteres.append("((dfComplete.cell=='%s') & (dfComplete.form=='%s'))"%(k,v))
        nbInput=len(selCriteres)
        exec("%s=%s"%("testComplete","|".join(selCriteres)),globals())
        if row.lexeme in testLexemes:
            print("|".join(selCriteres))
            display(dfComplete.loc[testComplete])
        lLexemes=dfComplete.loc[testComplete].lexeme.unique().tolist()
        notLexemes=[]
        for lexeme in lLexemes:
            nbComplete=len(dfComplete.loc[(testComplete)&(dfComplete.lexeme==lexeme)])
            if row.lexeme in testLexemes:
                display(dfComplete.loc[dfComplete.lexeme==lexeme])
                print(lexeme,nbComplete)
            if nbComplete<nbInput:
                notLexemes.append(lexeme)
                # print(row.lexeme,lexeme,nbInput,nbComplete)
        lLexemes=[l for l in lLexemes if l not in notLexemes]
        outLexemes[row.lexeme]=lLexemes
        if len(lLexemes)>1:
            print("plusieurs candidats",lLexemes)
            print(row.dropna())
        elif len(lLexemes)==0:
            print("pas de candidat",row.dropna())
    return outLexemes

In [13]:
# outLexemes

# Calcul des performances

In [14]:
def assembleTriplet(triplet):
    result={}
    for k,v in triplet.items():
        result[k[1]]=v
    return result
    

def compareGold(predLexeme,goldLexeme,dfGold):
    positive={}
    negative={}
    missing={}
    
    (_,goldCells),(_,goldForms)=dfGold.loc[dfGold.lexeme==goldLexeme][["cell","form"]].sort_values("cell").to_dict().items()
    (_,predCells),(_,predForms)=dfOutput.loc[dfOutput.lexeme==predLexeme][["cell","form"]].sort_values("cell").to_dict().items()
    
    for k,goldCell in goldCells.items():
        for kk,predCell in predCells.items():
            if predCell==goldCell and predForms[kk]==goldForms[k]:
                positive[(k,goldCell)]=goldForms[k]
                break
                
    for k,goldCell in goldCells.items():
        if (k,goldCell) not in positive:
            for kk,predCell in predCells.items():
                if predCell==goldCell:
                    negative[(k,goldCell)]=predForms[kk]+"≠"+goldForms[k]
                    break
                    
    for k,goldCell in goldCells.items():
        if (k,goldCell) not in positive and (k,goldCell) not in negative:
            missing[(k,goldCell)]="Ø≠"+goldForms[k]
    return assembleTriplet(positive),assembleTriplet(negative),assembleTriplet(missing)


In [15]:
def evaluations(dfCompare):
    connu={}
    correct={}
    different={}
    missing={}
    for predLexeme,checks in dict(list(outLexemes.items())[:]).items():
        if len(checks)>1 or "amalgamer" in checks: print(predLexeme,checks)
        connu[predLexeme]=dfInput.loc[dfInput.lexeme==predLexeme].set_index("cell")["form"].to_dict()
        maxPositive=0
        minNegative=len(sampleCases)
        for goldLexeme in checks:
            lPositive,lNegative,lMissing=compareGold(predLexeme,goldLexeme,dfCompare)
            if len(lPositive)>maxPositive:
                maxPositive=len(lPositive)
                correct[predLexeme]=lPositive
                different[predLexeme]=lNegative
                missing[predLexeme]=lMissing
            elif len(lPositive)==maxPositive:
                if len(lNegative)<minNegative:
                    minNegative=len(lNegative)
                    correct[predLexeme]=lPositive
                    different[predLexeme]=lNegative
                    missing[predLexeme]=lMissing
    return connu,correct,different,missing

In [16]:
def storeResults(df,checkType,stemType):

    def countForms(dForms,name):
        df=pd.DataFrame.from_dict(dForms)
        df.T.to_csv(repFiles+"vlexique2-%s%d%s-%s-%s.csv"%(inputType,num,stemType,checkType,name),sep=";")
        nb=df[df.notnull()].count().sum()
        return int(nb)
    
    connu,correct,different,missing=evaluations(df)
    nbConnu=countForms(connu,"input")
    nbCorrect=countForms(correct,"correct")
    nbDifferent=countForms(different,"different")
    nbMissing=countForms(missing,"missing")
    if (nbCorrect+nbDifferent)>0:
        precision=float(nbCorrect)/(nbCorrect+nbDifferent)*100
    else:
        precision=np.NaN
    if (nbCorrect+nbMissing)>0:
        rappel=float(nbCorrect)/(nbCorrect+nbMissing)*100
    else:
        rappel=np.NaN
    print("input :",inputFile,"output :",outputFile,"check type :",checkType)
    print("Brut précision %.1f, rappel %.1f"%(precision,rappel))

    with open(fStatistiques,"r") as inFile:
         stats=yaml.safe_load(inFile)
    if not stats:
        stats={}
    refStats=inputType+str(num)+stemType
    if refStats not in stats:
        stats[refStats]={}
    stats[refStats][checkType]={}
    stats[refStats][checkType]["nbConnu"]=nbConnu
    stats[refStats][checkType]["nbCorrect"]=nbCorrect
    stats[refStats][checkType]["nbDifferent"]=nbDifferent
    stats[refStats][checkType]["nbMissing"]=nbMissing
    stats[refStats][checkType]["precision"]=precision
    stats[refStats][checkType]["rappel"]=rappel
    
    yamlDump(fStatistiques,stats)    


In [17]:
def yamlDump(nFile,content):
    with open(nFile, 'w') as output:
        yaml.dump(content, output, default_flow_style=False,allow_unicode=True)

    with open(nFile, 'r') as input:
        yamlLines=input.readlines()

    yamlText="".join(yamlLines)
    yamlText=re.sub(r"!!python/unicode","",yamlText)
    yamlText=re.sub(r"\n\s*-\s*",", ",yamlText)
    yamlText=re.sub(r":,\s*",": ",yamlText)

    with open(nFile, 'w') as output:
        output.write(yamlText)
    return

In [19]:
repFiles="/Users/gilles/sDrive/Recherche/Boye/HDR/Data/vlexique2.0.3/WordStructure/"
platinumFile="vlexique2-Total.csv"
statistiquesFile="vlexique2-Statistiques.yaml"

inputTypes=["CV10-Train","CV10-Test"]
# stemTypes=["","-StemSpace"]
stemTypes=["-omp"]
nums=range(10)
# nums=[8]

for inputType in inputTypes:
    for inputNum in nums:
        num=inputNum
        for stemType in stemTypes:
            if inputType=="S":
                num=8-inputNum
                inputFile="vlexique2-S%d.csv"%num
                if stemType=="-omp":
                    outputFile="vlexique2-S%d%s.csv"%(num,stemType)
                else:
                    outputFile="vlexique2-S%d-omp-Swim2%s.csv"%(num,stemType)
                goldFile="vlexique2-R%d.csv"%num
            elif "CV" in inputType:
                quant,known=inputType.split("-")
                if known=="Train":
                    predict="Test"
                elif known=="Test":
                    predict="Train"
                inputFile="vlexique2-%s-%s%d.csv"%(quant,known,num)
                if stemType=="-omp":
                    outputFile="vlexique2-%s-%s%d%s.csv"%(quant,known,num,stemType)
                else:
                    outputFile="vlexique2-%s-%s%d-omp-Swim2%s.csv"%(quant,known,num,stemType)
                goldFile="vlexique2-%s-%s%d.csv"%(quant,predict,num)
            print(inputType,num,stemType) 
            
            fInput=repFiles+inputFile
            fOutput=repFiles+outputFile
            fGold=repFiles+goldFile
            fPlatinum=repFiles+platinumFile
            fStatistiques=repFiles+statistiquesFile
        
        
            tInput,dfInput=getTableDf(fInput)
            sampleCases=tInput.columns.values.tolist()
            sampleCases.remove(u"lexeme")
            
            # analyseCases=sampleCases
            for case in sampleCases:
                tInput[case]=tInput[case].apply(lambda x: checkFrench(x))
            countInput=tInput[sampleCases].stack().value_counts(dropna=True).sum()
            print("nombre de formes de départ",countInput)
            
            
            tOutPut,dfOutput=getTableDf(fOutput)
            dfOutput=separateData(dfOutput,dfInput).reset_index().drop("index", axis=1)
            # display(dfOutput)
            
            tGold,dfGold=getTableDf(fGold)
            dfGold=separateData(dfGold,dfInput).reset_index().drop("index", axis=1)
            countGoldPredictions=dfGold.loc[dfGold.cell.isin(sampleCases)].value_counts(dropna=True).sum()
            print("nombre de formes à générer pour Gold",countGoldPredictions)
            # display(dfGold)
            
            tComplete,dfComplete=getTableDf(fPlatinum)
            for case in sampleCases:
                tComplete[case]=tComplete[case].apply(lambda x: checkFrench(x))
            
            dfPlatinum=separateData(dfComplete,dfInput).reset_index().drop("index", axis=1)
            countPlatinumPredictions=dfPlatinum.loc[dfPlatinum.cell.isin(sampleCases)].value_counts(dropna=True).sum()
            print("nombre de formes à générer pour Platinum",countPlatinumPredictions)
            # display(dfPlatinum)
        
            outLexemes=identifyLexemes()
    
            storeResults(dfGold,"gold",stemType)
            storeResults(dfPlatinum,"platinum",stemType)   

CV10-Train 0 -omp
nombre de formes de départ 98134
nombre de formes à générer pour Gold 10917
nombre de formes à générer pour Platinum 170254
0, 100, 200, 300, plusieurs candidats ['bailler', 'bayer']
lexeme    bailler
ai3S         baja
ii1S         bajE
ii2S         bajE
ii3P         bajE
ii3S         bajE
inf          bajE
pI2P         bajE
pI2S          baj
pP           bajâ
pi1S          baj
pi2P         bajE
pi3P          baj
pi3S          baj
ppMS         bajE
ps3P          baj
ps3S          baj
Name: 393, dtype: object
400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, plusieurs candidats ['dépourvoir', 'pourvoir']
lexeme    dépourvoir
ai3S           purvy
inf          purvwar
ppFS           purvy
ppMP           purvy
ppMS           purvy
Name: 1584, dtype: object
1600, 1700, 1800, 1900, plusieurs candidats ['enter', 'hanter']
lexeme    enter
ai3P       âtEr
ai3S        âta
fi3S      ât9ra
ii3P        âtE
ii3S        âtE
inf         âtE
pI2P        âtE
pP         